# Linear combination of sources

`LinearCombination` builds one log-return series from **sources**:

`r[t] = constant * dt[t] + sum_i weights[i] * r_i[t] + sigma * sqrt(dt[t]) * eps[t]`

A source is a model (simulated by Fantasma), `(model, "X")` for one asset of a multi-asset model, a log-return Series (e.g. real prices converted with `fm.from_prices`), or another combination. Fantasma simulates the source models itself, and the result gives access to each source's realisation: its log returns and hidden state.

Fantasma simulates **log returns**; `fm.to_prices` turns them into prices.

Edit the parameters cell and re-run all: the same parameters and seeds always give the same paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import fantasma as fm

## Parameters

`weights` has one number per source. `constant` (annualised drift) and `sigma` (annualised volatility) enter each row as `constant * dt` and `sigma * sqrt(dt) * eps`, with `dt` the time since the previous row in years.

In [ ]:
# Sources: two models, simulated by Fantasma ...
MARKET = fm.RandomWalk(sigma=0.2, asset="MKT")
MOMENTUM = fm.StochasticDrift(sigma=0.1, drift_mean=0.05, drift_half_life=2.0, drift_sigma=0.3, asset="MOM")

# ... and one data series of log returns. Here it is simulated; with real data, e.g. from
# Almanac: fm.from_prices(load(...)["close"].unstack("symbol"))["SPY"]. With data among the
# sources, the grid is the data's index: the models are simulated on it.
DATA_SEED = 1
START = "2024-01-01"
FREQ = "1d"
N_STEPS = 365 * 3

# The combination
WEIGHTS = [1.2, 0.5, -0.3]  # MKT, MOM, SPY
SIGMA = 0.05
CONSTANT = 0.0  # annualised drift
INITIAL_PRICE = 100.0

# Simulation
SEED = 42
N_PATHS = 3

## Simulate

In [ ]:
(spy,) = fm.simulate(fm.RandomWalk(0.15, asset="SPY"), start=START, freq=FREQ, n_steps=N_STEPS, seed=DATA_SEED)
spy = spy["SPY"]  # a log-return Series

model = fm.LinearCombination(
    [MARKET, MOMENTUM, spy],
    weights=WEIGHTS,
    sigma=SIGMA,
    asset="STOCK",
    constant=CONSTANT,
    initial_price=INITIAL_PRICE,
)
model

In [ ]:
paths = fm.simulate(model, seed=SEED, n_paths=N_PATHS, return_hidden=True)
res = paths[0]
res.output.head()  # log returns of the combination

## The sources' realisations

`res.sources[name]` holds each source's log returns (`.output`) and hidden state (`.hidden`, on the full grid). Here: the market path, the momentum model with its true drift, and the data.

In [ ]:
print(list(res.sources))
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for name, src in res.sources.items():
    px = fm.to_prices(src.output)
    ax1.plot(px.index, px[name], lw=0.8, alpha=0.7, label=name)
px = fm.to_prices(res.output)
ax1.plot(px.index, px["STOCK"], lw=1, color="k", label="STOCK (path 0)")
ax1.set_ylabel("price")
ax1.legend(fontsize=8)
drift = res.sources["MOM"].hidden["drift.mu"]
ax2.plot(drift.index, drift, lw=0.8)
ax2.set_title("hidden: true drift of MOM")
plt.show()

## Check: regression recovers the parameters

OLS of the output on its sources' log returns, with a constant: the slopes estimate the weights, the intercept over `dt` the drift (noisy unless the sample is long), and the residual standard deviation over `sqrt(dt)` estimates `sigma`.

In [ ]:
for k, r in enumerate(paths):
    X = np.column_stack([np.ones(len(r.output))] + [r.sources[n].output[n] for n in ["MKT", "MOM", "SPY"]])
    beta, *_ = np.linalg.lstsq(X, r.output["STOCK"].to_numpy(), rcond=None)
    dt = fm.IndexGrid(spy.index, start=spy.attrs["start"]).dts
    resid = r.output["STOCK"].to_numpy() - X @ beta
    sigma_hat = resid.std(ddof=X.shape[1]) / np.sqrt(dt.mean())
    print(f"path {k}: weights {WEIGHTS} -> {np.round(beta[1:], 3)}, "
          f"constant {CONSTANT:+.3f} -> {beta[0] / dt.mean():+.3f}, sigma {SIGMA:.3f} -> {sigma_hat:.3f}")

## Several outputs sharing a source

Combinations simulated together share every model object they have in common: here two stocks load on the **same** market path. `fm.simulate([a, b], ...)` returns one frame with a column per output.

In [ ]:
a = fm.LinearCombination([MARKET], weights=[1.3], sigma=0.10, asset="A")
b = fm.LinearCombination([MARKET], weights=[0.7], sigma=0.15, asset="B")
two = fm.simulate([a, b], start=START, freq=FREQ, n_steps=N_STEPS, seed=SEED, return_hidden=True)[0]
print("correlation of A and B:", round(two.output.corr().iloc[0, 1], 3))
px = fm.to_prices(two.output)
mkt = fm.to_prices(two.sources["MKT"].output)
ax = plt.figure(figsize=(10, 3)).gca()
ax.plot(mkt.index, mkt["MKT"], color="k", lw=1, label="MKT (shared)")
for col in px.columns:
    ax.plot(px.index, px[col], lw=0.8, label=col)
ax.legend(fontsize=8)
plt.show()